In [1]:
from itertools import product
import os

In [18]:
class CNF:
    def __init__(self):
        self.clauses = []
        self.number_to_var_name = {}
        self.var_name_to_number = {}
    
    def add_clause(self, clause):
        for literal in clause:
            var_name = literal.strip('-')
            if var_name not in self.var_name_to_number:
                var_number = len(self.var_name_to_number) + 1
                self.var_name_to_number[var_name] = var_number
                self.number_to_var_name[var_number] = var_name
        self.clauses.append(clause)

    def dimacs(self):
        result = f'p cnf {len(self.number_to_var_name)} {len(self.clauses)}\n'
        for clause in self.clauses:
            for literal in clause:
                if literal[0] == '-':
                    result += '-'
                result += f'{self.var_name_to_number[literal.strip('-')]} '
            result += '0\n'
        return result
    
    def get_var_name(self, number: int):
        return self.vars[number]

    def get_var_number(self, name: str):
        return self.var_name_to_number[name]

In [6]:
def minisat_solve(problem_name, problem_dimacs, number_to_var):
    with open(f'{problem_name}.cnf', 'w') as handle:
        handle.write(problem_dimacs)
    os.system(f'minisat {problem_name}.cnf {problem_name}_result.cnf')

    with open(f'{problem_name}_result.cnf', 'r') as result_file:
        lines = result_file.readlines()

    if lines[0].startswith('SAT'):
        print('SAT')
        var_values = {}
        for var in lines[1].split(' ')[:-1]:
            var_number = int(var.strip('-'))
            var_name = number_to_var[var_number]
            var_values[var_name] = 0 if var.startswith('-') else 1
        true_vars = list(filter(lambda v: v[1] == 1, var_values.items()))
        true_vars.sort()
        for var in true_vars:
            print(var)
    else:
        print('UNSAT')

In [20]:
def solve(n, fiksne, zabranjeno):
    cnf = CNF()

    # U svakom redu moze da se nadje jedna dama (posto je disjunkcija onda samo treba navesti sva polja u redu i to znaci da se nalazi na jednom od njih) 
    for i in range(n):
        clause = [f'q_{i}_{j}' for j in range(n)]
        cnf.add_clause(clause)

    # Mora biti samo jedna dama po redu
    for i in range(n):
        for j in range(n-1):
            for k in range(j+1, n):
                cnf.add_clause([f'-q_{i}_{j}', f'-q_{i}_{k}'])

    # Mora biti samo jedna dama po koloni
    for i in range(n):
        for j in range(n-1):
            for k in range(j+1, n):
                cnf.add_clause([f'-q_{j}_{i}', f'-q_{k}_{i}'])

    # Moze se naci najvise jedna dama po dijagonali
    for i,j,k,l in product(range(n), repeat=4):
        if k > i and abs(k - i) == abs(l - j):
            cnf.add_clause([f'-q_{i}_{j}',f'-q_{k}_{l}'])

    # pretpostavljam da je niz tuple-a
    for (a, b) in fiksne:
        cnf.add_clause(f'q_{a}_{b}')

    for (a, b) in zabranjeno:
        cnf.add_clause(f'-q_{a}_{b}')
    
    minisat_solve(f'{n}_queens', cnf.dimacs(), cnf.number_to_var_name)

In [21]:
solve(4, [], [])

============================[ Problem Statistics ]=============================
|                                                                             |
|  Number of variables:            16                                         |
|  Number of clauses:              80                                         |
|  Parse time:                   0.00 s                                       |
|  Eliminated clauses:           0.00 Mb                                      |
|  Simplification time:          0.00 s                                       |
|                                                                             |
============================[ Search Statistics ]==============================
| Conflicts |          ORIGINAL         |          LEARNT          | Progress |
|           |    Vars  Clauses Literals |    Limit  Clauses Lit/Cl |          |
restarts              : 1
conflicts             : 0              (0 /sec)
decisions             : 5              (0.00 %